### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="diabetes_130_us",
    dataset_year="2014",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5230J",
    download_description="""
We download the data from the UCI repository and unzip it to a predefined folder.

mkdir -p local-data-warehouse/diabetes_130_us/ && wget -P local-data-warehouse/diabetes_130_us/ https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip && unzip local-data-warehouse/diabetes_130_us/diabetes+130-us+hospitals+for+years+1999-2008.zip -d local-data-warehouse/diabetes_130_us/
""",
    # References
    academic_reference_bibtex="""@article{strack2014impact,
  title={Impact of HbA1c measurement on hospital readmission rates: analysis of 70,000 clinical database patient records},
  author={Strack, Beata and DeShazo, Jonathan P and Gennings, Chris and Olmo, Juan L and Ventura, Sebastian and Cios, Krzysztof J and Clore, John N},
  journal={BioMed research international},
  volume={2014},
  number={1},
  pages={781670},
  year={2014},
  publisher={Wiley Online Library}
}
""",
    academic_reference_bibtex_key="strack2014impact",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We drop duplicated patients based on the "patient_nbr" feature to avoid target leakage.
- We reversed the original ordinal encoding for three ID-based features (admission_type_id, discharge_disposition_id, admission_source_id).
- We created the target from the "readmitted" column following the original task description: "<30" becomes "Yes", everything else "No".
- We dropped "encounter_id" and "patient_nbr", which are both unique identifiers for each row.
- We keep original "?", NULL-codes, and NaN values because they exist in different ways across the columns.
- Anomaly: There is a distribution shift based on the original order. The reason for this might be that the encounters are ordered in some way such that later parts of the data contain different sub-groups than earlier parts. This is also indicated by the fact that the "payer_code" feature is responsible for the shift. This distribution shift vanishes after randomly shuffling the data (as done by default for this and all other datasets used in TabArena).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="EarlyReadmission",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="EarlyReadmission",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/diabetic_data.csv")

# Drop duplicate patients
df = df.drop_duplicates(subset="patient_nbr")
# Drop identifier columns
df = df.drop(columns=["encounter_id", "patient_nbr"])

# Reverse ordinal encoding for ID-based features
inverse_mappings = {
    "admission_type_id": {
        1: "Emergency",
        2: "Urgent",
        3: "Elective",
        4: "Newborn",
        5: "Not Available",
        6: "NULL",
        7: "Trauma Center",
        8: "Not Mapped",
    },
    "discharge_disposition_id": {
        1: "Discharged to home",
        2: "Discharged/transferred to another short term hospital",
        3: "Discharged/transferred to SNF",
        4: "Discharged/transferred to ICF",
        5: "Discharged/transferred to another type of inpatient care institution",
        6: "Discharged/transferred to home with home health service",
        7: "Left AMA",
        8: "Discharged/transferred to home under care of Home IV provider",
        9: "Admitted as an inpatient to this hospital",
        10: "Neonate discharged to another hospital for neonatal aftercare",
        11: "Expired",
        12: "Still patient or expected to return for outpatient services",
        13: "Hospice / home",
        14: "Hospice / medical facility",
        15: "Discharged/transferred within this institution to Medicare approved swing bed",
        16: "Discharged/transferred/referred another institution for outpatient services",
        17: "Discharged/transferred/referred to this institution for outpatient services",
        18: "NULL",
        19: "Expired at home. Medicaid only, hospice.",
        20: "Expired in a medical facility. Medicaid only, hospice.",
        21: "Expired, place unknown. Medicaid only, hospice.",
        22: "Discharged/transferred to another rehab fac including rehab units of a hospital .",
        23: "Discharged/transferred to a long term care hospital.",
        24: "Discharged/transferred to a nursing facility certified under Medicaid but not certified under Medicare.",
        25: "Not Mapped",
        26: "Unknown/Invalid",
        27: "Discharged/transferred to a federal health care facility.",
        28: "Discharged/transferred/referred to a psychiatric hospital of psychiatric distinct part unit of a hospital",
        29: "Discharged/transferred to a Critical Access Hospital (CAH).",
        30: "Discharged/transferred to another Type of Health Care Institution not Defined Elsewhere",
    },
    "admission_source_id": {
        1: "Physician Referral",
        2: "Clinic Referral",
        3: "HMO Referral",
        4: "Transfer from a hospital",
        5: "Transfer from a Skilled Nursing Facility (SNF)",
        6: "Transfer from another health care facility",
        7: "Emergency Room",
        8: "Court/Law Enforcement",
        9: "Not Available",
        10: "Transfer from critial access hospital",
        11: "Normal Delivery",
        12: "Premature Delivery",
        13: "Sick Baby",
        14: "Extramural Birth",
        15: "Not Available",
        17: "NULL",
        18: "Transfer From Another Home Health Agency",
        19: "Readmission to Same Home Health Agency",
        20: "Not Mapped",
        21: "Unknown/Invalid",
        22: "Transfer from hospital inpt/same fac reslt in a sep claim",
        23: "Born inside this hospital",
        24: "Born outside this hospital",
        25: "Transfer from Ambulatory Surgery Center",
        26: "Transfer from Hospice",
    },
}
for feature, inverse_map in inverse_mappings.items():
    df[feature] = df[feature].map(inverse_map)

# Create binary target
target_feature = "EarlyReadmission"
df = df.rename(columns={"readmitted": target_feature})
label_mask = df[target_feature] == "<30"
df.loc[label_mask, target_feature] = "Yes"
df.loc[~label_mask, target_feature] = "No"

cat_features = [
    "race",
    "gender",
    "age",
    "weight",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "payer_code",
    "medical_specialty",
    "diag_1",
    "diag_2",
    "diag_3",
    "max_glu_serum",
    "A1Cresult",
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "acetohexamide",
    "glipizide",
    "glyburide",
    "tolbutamide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "troglitazone",
    "tolazamide",
    "examide",
    "citoglipton",
    "insulin",
    "glyburide-metformin",
    "glipizide-metformin",
    "glimepiride-pioglitazone",
    "metformin-rosiglitazone",
    "metformin-pioglitazone",
    "change",
    "diabetesMed",
    "EarlyReadmission",
]

# Shuffle to remove distribution shift from original order
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category")

# Drop constant features
df = df.drop(columns=[
    "examide",
    "citoglipton",
    "glimepiride-pioglitazone"
])

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 71,518
Columns: 45
Use sampling: False (sample size: 71,518)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['diag_3', 'diag_2', 'diag_1', 'num_lab_procedures', 'num_medications', 'medical_specialty', 'number_outpatient', 'discharge_disposition_id', 'number_emergency', 'payer_code']
Rows remaining as candidates after top-10 filter: 88 (of 71,518)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,insulin,glyburide-metformin,glipizide-metformin,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,EarlyReadmission
0,AfricanAmerican,Male,[60-70),?,Emergency,Discharged to home,Emergency Room,10,MC,Family/GeneralPractice,48,4,32,0,0,0,552,496,295,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No
1,Caucasian,Female,[50-60),?,Emergency,Discharged to home,Emergency Room,1,?,?,35,0,13,0,0,0,414,411,412,6,NaN,NaN,Steady,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,Ch,Yes,No
2,AfricanAmerican,Male,[70-80),?,Urgent,Discharged to home,Physician Referral,2,MC,Neurology,42,1,8,0,0,0,996,276,403,5,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No
3,AfricanAmerican,Female,[60-70),?,Elective,Discharged to home,Physician Referral,2,MC,Family/GeneralPractice,38,6,18,1,0,0,414,V45,250,8,NaN,NaN,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Steady,No,No,No,No,Ch,Yes,Yes
4,Caucasian,Female,[30-40),?,Elective,Discharged to home,Physician Referral,4,HM,?,33,3,20,0,0,0,661,648,657,6,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,max_glu_serum,category,68062.0,95.17,3.0,"Norm, >200, >300"
1,A1Cresult,category,58532.0,81.84,3.0,">8, Norm, >7"
2,race,category,0.0,0.00,6.0,"Caucasian, AfricanAmerican, ?, Hispanic, Other, Asian"
3,gender,category,0.0,0.00,3.0,"Female, Male, Unknown/Invalid"
4,age,category,0.0,0.00,10.0,"[70-80), [60-70), [50-60), [80-90), [40-50), [30-40), [90-100), [20-30), [10-20), [0-10)"
5,weight,category,0.0,0.00,10.0,"?, [75-100), [50-75), [100-125), [125-150), [25-50), [0-25), [150-175), [175-200), >200"
6,admission_type_id,category,0.0,0.00,8.0,"Emergency, Elective, Urgent, NULL, Not Available, Not Mapped, Trauma Center, Newborn"
7,discharge_disposition_id,category,0.0,0.00,26.0,"Discharged to home, Discharged/transferred to SNF, Discharged/transferred to home with home health service, NULL, Discharged/transferred to another short term hospital, Discharged/transferred to another rehab fac including rehab units of a hospital ., Expired, Discharged/transferred to another type of inpatient care institution, Not Mapped, Discharged/transferred to ICF"
8,admission_source_id,category,0.0,0.00,17.0,"Emergency Room, Physician Referral, NULL, Transfer from a hospital, Transfer from another health care facility, Clinic Referral, Transfer from a Skilled Nursing Facility (SNF), Not Mapped, HMO Referral, Not Available"
9,payer_code,category,0.0,0.00,18.0,"?, MC, HM, BC, SP, MD, CP, UN, CM, OG"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
time_in_hospital,71518.0,4.289130,2.949210,1.0,14.0
num_lab_procedures,71518.0,43.075478,19.952338,1.0,132.0
num_procedures,71518.0,1.430577,1.759864,0.0,6.0
num_medications,71518.0,15.705025,8.311163,1.0,81.0
number_outpatient,71518.0,0.280069,1.068957,0.0,42.0
number_emergency,71518.0,0.103540,0.509187,0.0,42.0
number_inpatient,71518.0,0.177829,0.603790,0.0,12.0
number_diagnoses,71518.0,7.245700,1.994674,1.0,16.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  \
column                   rank                                                            
A1Cresult                1                                                        <NA>   
                         2                                                          >8   
                         3                                                        Norm   
                         4                                                          >7   
EarlyReadmission         1                                                          No   
                         2                                                         Yes   
acarbose                 1                                                          No   
                         2                                                      Steady   
                         3                                                          Up   
acetohexamide            1                                                          No   
                         2                                                      Steady   
admission_source_id      1                                              Emergency Room   
                         2                                          Physician Referral   
                         3                                                        NULL   
                         4                                    Transfer from a hospital   
                         5                  Transfer from another health care facility   
admission_type_id        1                                                   Emergency   
                         2                                                    Elective   
                         3                                                      Urgent   
                         4                                                        NULL   
                         5                                               Not Available   
age                      1                                                     [70-80)   
                         2                                                     [60-70)   
                         3                                                     [50-60)   
                         4                                                     [80-90)   
                         5                                                     [40-50)   
change                   1                                                          No   
                         2                                                          Ch   
chlorpropamide           1                                                          No   
                         2                                                      Steady   
                         3                                                          Up   
                         4                                                        Down   
diabetesMed              1                                                         Yes   
                         2                                                          No   
diag_1                   1                                                         414   
                         2                                                         428   
                         3                                                         786   
                         4                                                         410   
                         5                                                         486   
diag_2                   1                                                         250   
                         2                                                         276   
                         3                                                         428   
                         4                                                         427   
                         5     

In [8]:
# Target Distribution
target_df

,count,pct
EarlyReadmission,,
No,65225,91.2
Yes,6293,8.8


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...


Saving curated container to diabetes_130_us/019d7367-64e4-7883-956d-9ba2fc8db41b


019d7367-64e4-7883-956d-9ba2fc8db41b
7022573ea873b486ca5b3e3a1b0ef7983300a32ef2c0ede2f32182bf1aea313e
